# Benchmark Model Training

### Logistic Regression, Random Forest, XGBoost


## Logistic Regression (Mean Rating)


In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))


from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER
import numpy as np

model_data = ModelData.from_pickle(PROCESSED_DATA_FOLDER / "processed_data.pkl")
model_data

In [ ]:
# Random Seed für Reproduzierbarkeit
SEED = 42

In [ ]:
# Aufteilung in business_covariates und rating_stats

# Business covariates (ohne Rating-Statistiken und OHNE Closed!)
business_features = [
    "density",
    "Checkin",
    "category",
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Distance.To.City.Centre",
    "Age",
]

# Rating statistics
rating_features = [
    "VAR",
    "MEAN",
    "ENTR",
    "COUNT",
    "ONE_STAR",
    "TWO_STAR",
    "THREE_STAR",
    "FOUR_STAR",
    "FIVE_STAR",
    "l_COUNT",
]

print(f"Business features: {len(business_features)} Variablen")
print(f"Rating features: {len(rating_features)} Variablen")
print(f"Gesamt: {len(business_features) + len(rating_features)} Variablen")

# Train/Test Split basierend auf train_indices
train_indices = model_data.train_indices
test_indices = np.array(
    [i for i in range(len(model_data.benchmark_covariates)) if i not in train_indices]
)
eval_indices = model_data.eval_indices
calibration_indices = model_data.calibration_indices

print(f"\nTrain/Test Split:")
print(f"  Training samples: {len(train_indices)}")
print(f"  Test samples: {len(eval_indices)}")
print(f"  Calibration samples: {len(calibration_indices)}")
print(f"  Total: {len(train_indices) + len(eval_indices) + len(calibration_indices)}")

## Datenvorbereitungen für alle Modelle


In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Target Variable vorbereiten
y = model_data.benchmark_covariates["Closed"].map({"Open": 0, "Closed": 1}).values

# ============================================================================
# 1. X_mean: Nur Mean Rating
# ============================================================================
X_mean = model_data.benchmark_covariates[["MEAN"]].values

# ============================================================================
# 2. X_business: Business Covariates (mit Preprocessing)
# ============================================================================
df_business = model_data.benchmark_covariates[business_features].copy()

# Kategorische Variablen identifizieren
categorical_cols = ["category"]
numeric_cols = [col for col in business_features if col not in categorical_cols]

# One-Hot Encoding
df_business_encoded = pd.get_dummies(
    df_business, columns=categorical_cols, drop_first=True
)

# Imputation und Standardisierung (nur auf Train-Daten fitten!)
imputer_business = SimpleImputer(strategy="median")
scaler_business = StandardScaler()

# Train-Daten preprocessing
numeric_train = df_business_encoded.iloc[train_indices][numeric_cols]
imputer_business.fit(numeric_train)
scaler_business.fit(imputer_business.transform(numeric_train))

# Transform für alle Daten
numeric_data_imputed = imputer_business.transform(df_business_encoded[numeric_cols])
numeric_data = scaler_business.transform(numeric_data_imputed)

# Kombiniere Features
X_business = pd.DataFrame(
    numeric_data, columns=numeric_cols, index=df_business_encoded.index
)
dummy_cols = [col for col in df_business_encoded.columns if col not in numeric_cols]
X_business = pd.concat([X_business, df_business_encoded[dummy_cols]], axis=1)

# ============================================================================
# 3. X_all: Alle Variablen (Business Covariates + Rating Stats)
# ============================================================================
all_features = business_features + rating_features

# Daten vorbereiten
df_all = model_data.benchmark_covariates[all_features].copy()

# Kategorische Variablen
categorical_cols = ["category"]
numeric_cols = [col for col in all_features if col not in categorical_cols]

# One-Hot Encoding
df_all_encoded = pd.get_dummies(df_all, columns=categorical_cols, drop_first=True)

# Imputation und Standardisierung (nur auf Train-Daten fitten!)
imputer_all = SimpleImputer(strategy="median")
scaler_all = StandardScaler()

# Train-Daten preprocessing
numeric_train = df_all_encoded.iloc[train_indices][numeric_cols]
imputer_all.fit(numeric_train)
scaler_all.fit(imputer_all.transform(numeric_train))

# Transform für alle Daten
numeric_data_imputed = imputer_all.transform(df_all_encoded[numeric_cols])
numeric_data = scaler_all.transform(numeric_data_imputed)

# Kombiniere Features
X_all = pd.DataFrame(numeric_data, columns=numeric_cols, index=df_all_encoded.index)
dummy_cols = [col for col in df_all_encoded.columns if col not in numeric_cols]
X_all = pd.concat([X_all, df_all_encoded[dummy_cols]], axis=1)

# ============================================================================
# Train/Test Split für alle Feature-Sets
# ============================================================================
X_mean_train, X_mean_test = X_mean[train_indices], X_mean[test_indices]
X_business_train = X_business.iloc[train_indices]
X_business_test = X_business.iloc[test_indices]
X_all_train = X_all.iloc[train_indices]
X_all_test = X_all.iloc[test_indices]

y_train, y_test = y[train_indices], y[test_indices]

# ============================================================================
# Zusammenfassung
# ============================================================================
print()
print("DATENVORBEREITUNGEN ABGESCHLOSSEN")
print("=" * 80)
print(f"\nTarget Variable (y):")
print(
    f"  Train samples: {len(y_train)} (Closed: {sum(y_train)}, Open: {len(y_train) - sum(y_train)})"
)
print(
    f"  Test samples:  {len(y_test)} (Closed: {sum(y_test)}, Open: {len(y_test) - sum(y_test)})"
)

print(f"\nFeature Sets:")
print(
    f"  X_mean shape:     {X_mean.shape} (Train: {X_mean_train.shape}, Test: {X_mean_test.shape})"
)
print(
    f"  X_business shape: {X_business.shape} (Train: {X_business_train.shape}, Test: {X_business_test.shape})"
)
print(
    f"  X_all shape:      {X_all.shape} (Train: {X_all_train.shape}, Test: {X_all_test.shape})"
)

print(f"\nFeature Details:")
print(f"  Mean Rating: 1 Variable")
print(
    f"  Business Covariates: {len(X_business.columns)} Variablen (nach One-Hot Encoding)"
)
print(f"  All Features: {len(X_all.columns)} Variablen (nach One-Hot Encoding)")

## Model 1: Logistic Regression mit Mean Rating


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Train Logistic Regression
lr_mean = LogisticRegression(random_state=SEED, max_iter=1000)
lr_mean.fit(X_mean_train, y_train)

# Predictions auf Train und Test
y_proba_mean_train = lr_mean.predict_proba(X_mean_train)[:, 1]
y_proba_mean_test = lr_mean.predict_proba(X_mean_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_mean_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_mean_test):.4f}")

# Koeffizienten
print(f"\nKoeffizient MEAN: {lr_mean.coef_[0][0]:.4f}")
print(f"Intercept: {lr_mean.intercept_[0]:.4f}")

## Model 2: Logistic Regression mit Business Covariates


In [ ]:
# Train Logistic Regression
lr_business = LogisticRegression(random_state=SEED, max_iter=1000)
lr_business.fit(X_business_train, y_train)

# Predictions
y_proba_business_train = lr_business.predict_proba(X_business_train)[:, 1]

y_proba_business_test = lr_business.predict_proba(X_business_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_business_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_business_test):.4f}")

# Top 10 wichtigste Features
coef_df = pd.DataFrame(
    {"feature": X_business.columns, "coefficient": lr_business.coef_[0]}
)
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 wichtigste Features:")
print(coef_df.head(10)[["feature", "coefficient"]])

## Model 3: Logistic Regression mit allen Variablen (Business + Rating Stats)


In [ ]:
# Train Logistic Regression
lr_all = LogisticRegression(random_state=SEED, max_iter=1000)
lr_all.fit(X_all_train, y_train)

# Predictions
y_proba_all_train = lr_all.predict_proba(X_all_train)[:, 1]
y_proba_all_test = lr_all.predict_proba(X_all_test)[:, 1]

print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_all_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_all_test):.4f}")

# Top 10 wichtigste Features
coef_df_all = pd.DataFrame({"feature": X_all.columns, "coefficient": lr_all.coef_[0]})
coef_df_all["abs_coefficient"] = coef_df_all["coefficient"].abs()
coef_df_all = coef_df_all.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 wichtigste Features:")
print(coef_df_all.head(10)[["feature", "coefficient"]])

## Vergleich der drei Benchmark-Modelle


In [ ]:
# Vergleichstabelle erstellen (TEST Performance!)
import pandas as pd

results_comparison = pd.DataFrame(
    {
        "Model": ["Mean Rating", "Business Covariates", "All Features"],
        "Features": [
            "MEAN (1 Variable)",
            f"{len(X_business.columns)} Variablen",
            f"{len(X_all.columns)} Variablen",
        ],
        "Train Accuracy": [
            accuracy_score(y_train, y_pred_mean_train),
            accuracy_score(y_train, y_pred_business_train),
            accuracy_score(y_train, y_pred_all_train),
        ],
        "Test Accuracy": [
            accuracy_score(y_test, y_pred_mean_test),
            accuracy_score(y_test, y_pred_business_test),
            accuracy_score(y_test, y_pred_all_test),
        ],
        "Train ROC-AUC": [
            roc_auc_score(y_train, y_proba_mean_train),
            roc_auc_score(y_train, y_proba_business_train),
            roc_auc_score(y_train, y_proba_all_train),
        ],
        "Test ROC-AUC": [
            roc_auc_score(y_test, y_proba_mean_test),
            roc_auc_score(y_test, y_proba_business_test),
            roc_auc_score(y_test, y_proba_all_test),
        ],
    }
)

print("=" * 80)
print("VERGLEICH DER BENCHMARK-MODELLE")
print("=" * 80)
print(results_comparison.to_string(index=False))

# Visualisierung
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Vergleich
x = np.arange(len(results_comparison))
width = 0.35

axes[0].bar(
    x - width / 2,
    results_comparison["Train Accuracy"],
    width,
    label="Train",
    color="#1f77b4",
    alpha=0.8,
)
axes[0].bar(
    x + width / 2,
    results_comparison["Test Accuracy"],
    width,
    label="Test",
    color="#ff7f0e",
    alpha=0.8,
)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title("Accuracy Vergleich (Train vs Test)", fontsize=14, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_comparison["Model"], rotation=15, ha="right")
axes[0].set_ylim([0, 1])
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# ROC-AUC Vergleich
axes[1].bar(
    x - width / 2,
    results_comparison["Train ROC-AUC"],
    width,
    label="Train",
    color="#1f77b4",
    alpha=0.8,
)
axes[1].bar(
    x + width / 2,
    results_comparison["Test ROC-AUC"],
    width,
    label="Test",
    color="#ff7f0e",
    alpha=0.8,
)
axes[1].set_ylabel("ROC-AUC", fontsize=12)
axes[1].set_title("ROC-AUC Vergleich (Train vs Test)", fontsize=14, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_comparison["Model"], rotation=15, ha="right")
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Bestes Modell identifizieren (basierend auf Test ROC-AUC)
best_model_idx = results_comparison["Test ROC-AUC"].idxmax()
print(
    f"\nBestes Modell (Test ROC-AUC): {results_comparison.loc[best_model_idx, 'Model']}"
)
print(f"Test ROC-AUC: {results_comparison.loc[best_model_idx, 'Test ROC-AUC']:.4f}")
print(f"Test Accuracy: {results_comparison.loc[best_model_idx, 'Test Accuracy']:.4f}")

# Random Forest Models

---


## Model 4: Random Forest mit Mean Rating


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Model 4: Random Forest mit Mean Rating
# Parameter aus Original-Paper (3_analysis.R, Zeile 509-518)
rf_mean = RandomForestClassifier(
    random_state=SEED, n_estimators=500, max_leaf_nodes=9  # ntree = 500  # maxnodes = 9
)
rf_mean.fit(X_mean_train, y_train)

# Predictions
y_proba_rf_mean_train = rf_mean.predict_proba(X_mean_train)[:, 1]
y_proba_rf_mean_test = rf_mean.predict_proba(X_mean_test)[:, 1]

# Evaluation
print("\nParameter (aus Original-Paper):")
print(f"  n_estimators: {rf_mean.n_estimators}")
print(f"  max_leaf_nodes: {rf_mean.max_leaf_nodes}")

print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_rf_mean_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_rf_mean_test):.4f}")

# Feature Importance
print(f"\nFeature Importance: MEAN = {rf_mean.feature_importances_[0]:.4f}")

## Model 5: Random Forest mit Business Covariates


In [ ]:
# Model 5: Random Forest mit Business Covariates
# Parameter aus Original-Paper (3_analysis.R, Zeile 509-518)
rf_business = RandomForestClassifier(
    random_state=SEED, n_estimators=500, max_leaf_nodes=9  # ntree = 500  # maxnodes = 9
)
rf_business.fit(X_business_train, y_train)

# Predictions
y_proba_rf_business_train = rf_business.predict_proba(X_business_train)[:, 1]
y_proba_rf_business_test = rf_business.predict_proba(X_business_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_rf_business_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_rf_business_test):.4f}")


# Top 10 Feature Importances
feature_importance_df = pd.DataFrame(
    {"feature": X_business.columns, "importance": rf_business.feature_importances_}
)
feature_importance_df = feature_importance_df.sort_values("importance", ascending=False)

print("\nTop 10 wichtigste Features:")
print(feature_importance_df.head(10))

## Model 6: Random Forest mit allen Variablen (Business + Rating Stats)


In [ ]:
# Model 6: Random Forest mit allen Variablen
# Parameter aus Original-Paper (3_analysis.R, Zeile 509-518)
rf_all = RandomForestClassifier(
    random_state=SEED, n_estimators=500, max_leaf_nodes=9  # ntree = 500  # maxnodes = 9
)
rf_all.fit(X_all_train, y_train)

# Predictions
y_proba_rf_all_train = rf_all.predict_proba(X_all_train)[:, 1]
y_proba_rf_all_test = rf_all.predict_proba(X_all_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_rf_all_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_rf_all_test):.4f}")

# Top 10 Feature Importances
feature_importance_df_all = pd.DataFrame(
    {"feature": X_all.columns, "importance": rf_all.feature_importances_}
)
feature_importance_df_all = feature_importance_df_all.sort_values(
    "importance", ascending=False
)

print("\nTop 10 wichtigste Features:")
print(feature_importance_df_all.head(10))

## Vergleich Random Forest Modelle


In [ ]:
# Vergleich Random Forest Modelle
rf_results = pd.DataFrame(
    {
        "Model": ["RF: Mean Rating", "RF: Business Covariates", "RF: All Features"],
        "Features": [
            "MEAN (1 Variable)",
            f"{len(X_business.columns)} Variablen",
            f"{len(X_all.columns)} Variablen",
        ],
        "Train Accuracy": [
            accuracy_score(y_train, y_pred_rf_mean_train),
            accuracy_score(y_train, y_pred_rf_business_train),
            accuracy_score(y_train, y_pred_rf_all_train),
        ],
        "Test Accuracy": [
            accuracy_score(y_test, y_pred_rf_mean_test),
            accuracy_score(y_test, y_pred_rf_business_test),
            accuracy_score(y_test, y_pred_rf_all_test),
        ],
        "Train ROC-AUC": [
            roc_auc_score(y_train, y_proba_rf_mean_train),
            roc_auc_score(y_train, y_proba_rf_business_train),
            roc_auc_score(y_train, y_proba_rf_all_train),
        ],
        "Test ROC-AUC": [
            roc_auc_score(y_test, y_proba_rf_mean_test),
            roc_auc_score(y_test, y_proba_rf_business_test),
            roc_auc_score(y_test, y_proba_rf_all_test),
        ],
    }
)

print("=" * 80)
print("VERGLEICH RANDOM FOREST MODELLE")
print("=" * 80)
print(rf_results.to_string(index=False))

# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(rf_results))
width = 0.35

# Accuracy
axes[0].bar(
    x - width / 2,
    rf_results["Train Accuracy"],
    width,
    label="Train",
    color="#2ca02c",
    alpha=0.8,
)
axes[0].bar(
    x + width / 2,
    rf_results["Test Accuracy"],
    width,
    label="Test",
    color="#d62728",
    alpha=0.8,
)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title(
    "Random Forest: Accuracy (Train vs Test)", fontsize=14, fontweight="bold"
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    ["Mean Rating", "Business Cov.", "All Features"], rotation=15, ha="right"
)
axes[0].set_ylim([0, 1])
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# ROC-AUC
axes[1].bar(
    x - width / 2,
    rf_results["Train ROC-AUC"],
    width,
    label="Train",
    color="#2ca02c",
    alpha=0.8,
)
axes[1].bar(
    x + width / 2,
    rf_results["Test ROC-AUC"],
    width,
    label="Test",
    color="#d62728",
    alpha=0.8,
)
axes[1].set_ylabel("ROC-AUC", fontsize=12)
axes[1].set_title(
    "Random Forest: ROC-AUC (Train vs Test)", fontsize=14, fontweight="bold"
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    ["Mean Rating", "Business Cov.", "All Features"], rotation=15, ha="right"
)
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Bestes Random Forest Modell
best_rf_idx = rf_results["Test ROC-AUC"].idxmax()
print(
    f"\nBestes Random Forest Modell (Test ROC-AUC): {rf_results.loc[best_rf_idx, 'Model']}"
)
print(f"Test ROC-AUC: {rf_results.loc[best_rf_idx, 'Test ROC-AUC']:.4f}")
print(f"Test Accuracy: {rf_results.loc[best_rf_idx, 'Test Accuracy']:.4f}")

# XGBoost Models

---

XGBoost als zusätzliches Benchmark-Modell (nicht im Original-Paper enthalten)


## Model 7: XGBoost mit Mean Rating


In [ ]:
from xgboost import XGBClassifier

# Model 7: XGBoost mit Mean Rating
xgb_mean = XGBClassifier(
    random_state=SEED,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric="logloss",
)
xgb_mean.fit(X_mean_train, y_train)

# Predictions
y_proba_xgb_mean_train = xgb_mean.predict_proba(X_mean_train)[:, 1]
y_proba_xgb_mean_test = xgb_mean.predict_proba(X_mean_test)[:, 1]

# Evaluation
print("=" * 60)
print("MODEL 7: XGBoost mit Mean Rating")
print("=" * 60)
print("\nParameter:")
print(f"  n_estimators: {xgb_mean.n_estimators}")
print(f"  max_depth: {xgb_mean.max_depth}")
print(f"  learning_rate: {xgb_mean.learning_rate}")

print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_xgb_mean_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_xgb_mean_test):.4f}")

# Feature Importance
print(f"\nFeature Importance: MEAN = {xgb_mean.feature_importances_[0]:.4f}")

## Model 8: XGBoost mit Business Covariates


In [ ]:
# Model 8: XGBoost mit Business Covariates
xgb_business = XGBClassifier(
    random_state=SEED,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric="logloss",
)
xgb_business.fit(X_business_train, y_train)

# Predictions
y_proba_xgb_business_train = xgb_business.predict_proba(X_business_train)[:, 1]
y_proba_xgb_business_test = xgb_business.predict_proba(X_business_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_xgb_business_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_xgb_business_test):.4f}")

# Top 10 Feature Importances
feature_importance_xgb_business = pd.DataFrame(
    {"feature": X_business.columns, "importance": xgb_business.feature_importances_}
)
feature_importance_xgb_business = feature_importance_xgb_business.sort_values(
    "importance", ascending=False
)

print("\nTop 10 wichtigste Features:")
print(feature_importance_xgb_business.head(10))

## Model 9: XGBoost mit allen Variablen


In [ ]:
# Model 9: XGBoost mit allen Variablen
xgb_all = XGBClassifier(
    random_state=SEED,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    eval_metric="logloss",
)
xgb_all.fit(X_all_train, y_train)

# Predictions
y_proba_xgb_all_train = xgb_all.predict_proba(X_all_train)[:, 1]
y_proba_xgb_all_test = xgb_all.predict_proba(X_all_test)[:, 1]

# Evaluation
print("\nTRAIN Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_train, y_proba_xgb_all_train):.4f}")

print("\nTEST Performance:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba_xgb_all_test):.4f}")

# Top 10 Feature Importances
feature_importance_xgb_all = pd.DataFrame(
    {"feature": X_all.columns, "importance": xgb_all.feature_importances_}
)
feature_importance_xgb_all = feature_importance_xgb_all.sort_values(
    "importance", ascending=False
)

print("\nTop 10 wichtigste Features:")
print(feature_importance_xgb_all.head(10))

## Vergleich XGBoost Modelle


In [ ]:
# Vergleich XGBoost Modelle
xgb_results = pd.DataFrame(
    {
        "Model": ["XGB: Mean Rating", "XGB: Business Covariates", "XGB: All Features"],
        "Features": [
            "MEAN (1 Variable)",
            f"{len(X_business.columns)} Variablen",
            f"{len(X_all.columns)} Variablen",
        ],
        "Train Accuracy": [
            accuracy_score(y_train, y_pred_xgb_mean_train),
            accuracy_score(y_train, y_pred_xgb_business_train),
            accuracy_score(y_train, y_pred_xgb_all_train),
        ],
        "Test Accuracy": [
            accuracy_score(y_test, y_pred_xgb_mean_test),
            accuracy_score(y_test, y_pred_xgb_business_test),
            accuracy_score(y_test, y_pred_xgb_all_test),
        ],
        "Train ROC-AUC": [
            roc_auc_score(y_train, y_proba_xgb_mean_train),
            roc_auc_score(y_train, y_proba_xgb_business_train),
            roc_auc_score(y_train, y_proba_xgb_all_train),
        ],
        "Test ROC-AUC": [
            roc_auc_score(y_test, y_proba_xgb_mean_test),
            roc_auc_score(y_test, y_proba_xgb_business_test),
            roc_auc_score(y_test, y_proba_xgb_all_test),
        ],
    }
)

print("=" * 80)
print("VERGLEICH XGBOOST MODELLE")
print("=" * 80)
print(xgb_results.to_string(index=False))

# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(xgb_results))
width = 0.35

# Accuracy
axes[0].bar(
    x - width / 2,
    xgb_results["Train Accuracy"],
    width,
    label="Train",
    color="#9467bd",
    alpha=0.8,
)
axes[0].bar(
    x + width / 2,
    xgb_results["Test Accuracy"],
    width,
    label="Test",
    color="#8c564b",
    alpha=0.8,
)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title("XGBoost: Accuracy (Train vs Test)", fontsize=14, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    ["Mean Rating", "Business Cov.", "All Features"], rotation=15, ha="right"
)
axes[0].set_ylim([0, 1])
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# ROC-AUC
axes[1].bar(
    x - width / 2,
    xgb_results["Train ROC-AUC"],
    width,
    label="Train",
    color="#9467bd",
    alpha=0.8,
)
axes[1].bar(
    x + width / 2,
    xgb_results["Test ROC-AUC"],
    width,
    label="Test",
    color="#8c564b",
    alpha=0.8,
)
axes[1].set_ylabel("ROC-AUC", fontsize=12)
axes[1].set_title("XGBoost: ROC-AUC (Train vs Test)", fontsize=14, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    ["Mean Rating", "Business Cov.", "All Features"], rotation=15, ha="right"
)
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Bestes XGBoost Modell
best_xgb_idx = xgb_results["Test ROC-AUC"].idxmax()
print(
    f"\nBestes XGBoost Modell (Test ROC-AUC): {xgb_results.loc[best_xgb_idx, 'Model']}"
)
print(f"Test ROC-AUC: {xgb_results.loc[best_xgb_idx, 'Test ROC-AUC']:.4f}")
print(f"Test Accuracy: {xgb_results.loc[best_xgb_idx, 'Test Accuracy']:.4f}")

# Finaler Gesamtvergleich

---

## Vergleich: Logistic Regression vs Random Forest (Original vs Tuned)


In [ ]:
# Import Helper-Funktionen
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))
from helpers.benchmark_helpers import create_model_comparison_df

# Finaler Gesamtvergleich aller Modelle
model_configs = [
    # Logistic Regression
    {
        "model_type": "Logistic Regression",
        "features": "Mean Rating",
        "y_pred_train": y_pred_mean_train,
        "y_proba_train": y_proba_mean_train,
        "y_pred_test": y_pred_mean_test,
        "y_proba_test": y_proba_mean_test,
    },
    {
        "model_type": "Logistic Regression",
        "features": "Business Cov.",
        "y_pred_train": y_pred_business_train,
        "y_proba_train": y_proba_business_train,
        "y_pred_test": y_pred_business_test,
        "y_proba_test": y_proba_business_test,
    },
    {
        "model_type": "Logistic Regression",
        "features": "All Features",
        "y_pred_train": y_pred_all_train,
        "y_proba_train": y_proba_all_train,
        "y_pred_test": y_pred_all_test,
        "y_proba_test": y_proba_all_test,
    },
    # Random Forest (Original)
    {
        "model_type": "Random Forest (Original)",
        "features": "All Features",
        "y_pred_train": y_pred_rf_all_train,
        "y_proba_train": y_proba_rf_all_train,
        "y_pred_test": y_pred_rf_all_test,
        "y_proba_test": y_proba_rf_all_test,
    },
    # Random Forest (Tuned)
    {
        "model_type": "Random Forest (Tuned)",
        "features": "All Features",
        "y_pred_train": y_pred_rf_all_tuned_train,
        "y_proba_train": y_proba_rf_all_tuned_train,
        "y_pred_test": y_pred_rf_all_tuned_test,
        "y_proba_test": y_proba_rf_all_tuned_test,
    },
    # XGBoost
    {
        "model_type": "XGBoost",
        "features": "Mean Rating",
        "y_pred_train": y_pred_xgb_mean_train,
        "y_proba_train": y_proba_xgb_mean_train,
        "y_pred_test": y_pred_xgb_mean_test,
        "y_proba_test": y_proba_xgb_mean_test,
    },
    {
        "model_type": "XGBoost",
        "features": "Business Cov.",
        "y_pred_train": y_pred_xgb_business_train,
        "y_proba_train": y_proba_xgb_business_train,
        "y_pred_test": y_pred_xgb_business_test,
        "y_proba_test": y_proba_xgb_business_test,
    },
    {
        "model_type": "XGBoost",
        "features": "All Features",
        "y_pred_train": y_pred_xgb_all_train,
        "y_proba_train": y_proba_xgb_all_train,
        "y_pred_test": y_pred_xgb_all_test,
        "y_proba_test": y_proba_xgb_all_test,
    },
]

# Erstelle Vergleichs-DataFrame mit Helper-Funktion
final_comparison = create_model_comparison_df(model_configs, y_train, y_test)

# Sortieren nach Test ROC-AUC
final_comparison_sorted = final_comparison.sort_values("Test ROC-AUC", ascending=False)

print("=" * 100)
print("FINALER GESAMTVERGLEICH ALLER MODELLE")
print("=" * 100)
print(final_comparison_sorted.to_string(index=False))

# Visualisierung
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Farben für verschiedene Modelltypen
colors_map = {
    "Logistic Regression": "#1f77b4",
    "Random Forest (Original)": "#ff7f0e",
    "Random Forest (Tuned)": "#2ca02c",
    "XGBoost": "#9467bd",
}
colors = [colors_map[mt] for mt in final_comparison["Model Type"]]

# 1. Test ROC-AUC Vergleich aller Modelle
model_labels = [
    f"{row['Model Type'][:20]}\n{row['Features'][:15]}"
    for _, row in final_comparison.iterrows()
]
test_roc_scores = final_comparison["Test ROC-AUC"].values

axes[0, 0].barh(range(len(model_labels)), test_roc_scores, color=colors, alpha=0.8)
axes[0, 0].set_yticks(range(len(model_labels)))
axes[0, 0].set_yticklabels(model_labels, fontsize=9)
axes[0, 0].set_xlabel("Test ROC-AUC", fontsize=12)
axes[0, 0].set_title(
    "Test ROC-AUC Vergleich aller Modelle", fontsize=14, fontweight="bold"
)
axes[0, 0].set_xlim([0, 1])
axes[0, 0].grid(axis="x", alpha=0.3)
axes[0, 0].invert_yaxis()

# Werte anzeigen
for i, v in enumerate(test_roc_scores):
    axes[0, 0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)

# 2. Overfitting Gap Vergleich
gap_values = final_comparison["Overfitting Gap"].values

axes[0, 1].barh(range(len(model_labels)), gap_values, color=colors, alpha=0.8)
axes[0, 1].set_yticks(range(len(model_labels)))
axes[0, 1].set_yticklabels(model_labels, fontsize=9)
axes[0, 1].set_xlabel("Overfitting Gap (Train - Test ROC-AUC)", fontsize=12)
axes[0, 1].set_title("Overfitting Gap Vergleich", fontsize=14, fontweight="bold")
axes[0, 1].axvline(x=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0, 1].grid(axis="x", alpha=0.3)
axes[0, 1].invert_yaxis()

# Werte anzeigen
for i, v in enumerate(gap_values):
    axes[0, 1].text(
        v + 0.005 if v > 0 else v - 0.005,
        i,
        f"{v:.3f}",
        va="center",
        ha="left" if v > 0 else "right",
        fontsize=8,
    )

# 3. Modelltyp-Vergleich (Best von jedem Typ)
best_per_type = final_comparison.loc[
    final_comparison.groupby("Model Type")["Test ROC-AUC"].idxmax()
]
best_labels = best_per_type["Model Type"].values
best_test = best_per_type["Test ROC-AUC"].values
best_train = best_per_type["Train ROC-AUC"].values
best_colors = [colors_map[mt] for mt in best_labels]

x = np.arange(len(best_labels))
width = 0.35

axes[1, 0].bar(
    x - width / 2, best_train, width, label="Train", alpha=0.6, color=best_colors
)
axes[1, 0].bar(
    x + width / 2, best_test, width, label="Test", alpha=0.8, color=best_colors
)
axes[1, 0].set_ylabel("ROC-AUC", fontsize=12)
axes[1, 0].set_title(
    "Beste Modelle pro Typ (Train vs Test)", fontsize=14, fontweight="bold"
)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(
    [label.replace(" ", "\n") for label in best_labels],
    rotation=0,
    ha="center",
    fontsize=9,
)
axes[1, 0].set_ylim([0, 1.05])
axes[1, 0].legend()
axes[1, 0].grid(axis="y", alpha=0.3)

# Werte über Balken
for i, (train, test) in enumerate(zip(best_train, best_test)):
    axes[1, 0].text(
        i - width / 2, train + 0.02, f"{train:.3f}", ha="center", fontsize=8
    )
    axes[1, 0].text(i + width / 2, test + 0.02, f"{test:.3f}", ha="center", fontsize=8)

# 4. Top 5 Modelle
top5 = final_comparison_sorted.head(5).copy()
top5_labels = [
    f"{row['Model Type'][:18]}\n{row['Features'][:12]}" for _, row in top5.iterrows()
]
top5_colors = [colors_map[mt] for mt in top5["Model Type"]]

axes[1, 1].barh(range(len(top5)), top5["Test ROC-AUC"], color=top5_colors, alpha=0.8)
axes[1, 1].set_yticks(range(len(top5)))
axes[1, 1].set_yticklabels(top5_labels, fontsize=9)
axes[1, 1].set_xlabel("Test ROC-AUC", fontsize=12)
axes[1, 1].set_title("Top 5 Modelle", fontsize=14, fontweight="bold")
axes[1, 1].set_xlim([0, 1])
axes[1, 1].grid(axis="x", alpha=0.3)
axes[1, 1].invert_yaxis()

# Werte anzeigen
for i, v in enumerate(top5["Test ROC-AUC"]):
    axes[1, 1].text(v + 0.01, i, f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

# Beste Modelle identifizieren
print("\n" + "=" * 100)
print("ZUSAMMENFASSUNG")
print("=" * 100)

best_overall = final_comparison_sorted.iloc[0]
print(f"\nBestes Modell insgesamt (Test ROC-AUC):")
print(f"  {best_overall['Model Type']} - {best_overall['Features']}")
print(f"  Test ROC-AUC: {best_overall['Test ROC-AUC']:.4f}")
print(f"  Test Accuracy: {best_overall['Test Accuracy']:.4f}")
print(f"  Overfitting Gap: {best_overall['Overfitting Gap']:.4f}")

# Beste Modelle pro Typ
print("\n" + "=" * 100)
print("BESTE MODELLE PRO TYP")
print("=" * 100)

for model_type in ["Logistic Regression", "Random Forest (Tuned)", "XGBoost"]:
    best = (
        final_comparison[final_comparison["Model Type"] == model_type]
        .sort_values("Test ROC-AUC", ascending=False)
        .iloc[0]
        if len(final_comparison[final_comparison["Model Type"] == model_type]) > 0
        else None
    )

    if best is not None:
        print(f"\n{model_type}:")
        print(f"  Features: {best['Features']}")
        print(f"  Test ROC-AUC: {best['Test ROC-AUC']:.4f}")
        print(f"  Test Accuracy: {best['Test Accuracy']:.4f}")
        print(f"  Overfitting Gap: {best['Overfitting Gap']:.4f}")

# Verbesserung durch Tuning (nur für All Features RF)
rf_orig = final_comparison[
    final_comparison["Model Type"] == "Random Forest (Original)"
].iloc[0]
rf_tuned = final_comparison[
    final_comparison["Model Type"] == "Random Forest (Tuned)"
].iloc[0]

print("\n" + "=" * 100)
print("EFFEKT DES HYPERPARAMETER-TUNINGS (Random Forest - All Features)")
print("=" * 100)

roc_improvement = rf_tuned["Test ROC-AUC"] - rf_orig["Test ROC-AUC"]
overfit_reduction = rf_orig["Overfitting Gap"] - rf_tuned["Overfitting Gap"]

print(
    f"\nTest ROC-AUC: {rf_orig['Test ROC-AUC']:.4f} → {rf_tuned['Test ROC-AUC']:.4f} ({roc_improvement:+.4f})"
)
print(
    f"Overfitting Gap: {rf_orig['Overfitting Gap']:.4f} → {rf_tuned['Overfitting Gap']:.4f} (Reduktion: {overfit_reduction:.4f})"
)
print(
    f"Test Accuracy: {rf_orig['Test Accuracy']:.4f} → {rf_tuned['Test Accuracy']:.4f} ({rf_tuned['Test Accuracy'] - rf_orig['Test Accuracy']:+.4f})"
)